In [2]:
import sys
import os

# Adjust the path to where the src folder is located
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

In [3]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from language_models.dictionary_corpus import Dictionary
from collections import defaultdict
import torch.nn as nn
from torch.nn.functional import scaled_dot_product_attention
import language_models.model as m
import math
import torch.nn.functional as F
from utils import WMTestDataset, collate_fn
from language_models.model import CBR_RNN

In [4]:
device = torch.device('cpu')

In [5]:
sentence_path = '/scratch2/mrenaudin/colorlessgreenRNNs/wm_tests/rnn_input_files/categorized_lists_sce3_repeat.txt'
marker_path = '/scratch2/mrenaudin/colorlessgreenRNNs/wm_tests/rnn_input_files/categorized_lists_sce3_repeat_markers.txt'
data_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data"
dictionary = Dictionary(data_path)
wm_dataset = WMTestDataset(sentence_path, marker_path, dictionary)
dataloader = DataLoader(wm_dataset, batch_size=230, collate_fn=collate_fn)

In [10]:
model = CBR_RNN(50001, 1024, 1024, 1, 0, device)
check = torch.load('/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/test_attention_1024_rest/epoch_30.pt', map_location='cpu')
model.load_state_dict(check['model_state_dict'])
temperature = check['temperature']

In [222]:
def eval_simplified(model, dataloader, temperature):

    model.eval()
    
    # Dictionary to track results by list length
    results_by_length = {}
    
    with torch.no_grad():
        for batch in dataloader:
            encoded_sentence = batch["encoded_sentence"]
            list1_encoded = batch['list1_encoded']
            list_len = list1_encoded.shape[-1]//2
            results_by_length[list_len]=[]
            marker = batch["marker"]
            sentence = batch['sentence']
            list2_word_pos = [i for i,m in enumerate(marker[0]) if m==3][::2]
           
            batch_size, seq_len = encoded_sentence.shape
            
            for i in range(1, len(list2_word_pos) - 1):
                word_pos = list2_word_pos[i]
                input_seq = encoded_sentence[:, :word_pos].transpose(0, 1)
                cache = model.init_cache(input_seq , 1)
                output, _ = model(input_seq, cache, 1, temperature, True)
                log_probs = torch.nn.functional.log_softmax(output, dim=-1)
                lp = log_probs[-1,:,:]
                # Compare correct vs. incorrect log probs
                correct_tokens = list1_encoded[:, i+1].unsqueeze(1)
                log_prob_correct = lp.gather(1, correct_tokens)

                acc = 0
                total = 0
                for j in range(list1_encoded.shape[1]):
                    if j == i:
                        continue
                    wrong_tokens = list1_encoded[:, j]
                    log_prob_wrong = lp.gather(1, wrong_tokens.unsqueeze(1))
                    acc += (log_prob_correct > log_prob_wrong).float().sum().item()
                    total += correct_tokens.size(0)

                accuracy = acc / total if total > 0 else 0.0
                results_by_length[list_len].append(accuracy)

    return results_by_length        
            

In [223]:
def eval_simplified(model, dataloader, temperature):
    model.eval()
    
    results_by_length = {}
    
    with torch.no_grad():
        for batch in dataloader:
            encoded_sentence = batch["encoded_sentence"]       # shape: (batch_size, seq_len)
            list1_encoded = batch['list1_encoded'][:,::2]   
            list_len = list1_encoded.shape[-1] 
            results_by_length.setdefault(list_len, [])
            
            marker = batch["marker"]
            list2_word_pos = [i for i,m in enumerate(marker[0]) if m==3][::2]
            
            batch_size, seq_len = encoded_sentence.shape
            
            # We'll keep track of correctness per sequence in the batch
            correct_sequences = torch.ones(batch_size, dtype=torch.bool)  # Start with all True
            
            for i in range(1, len(list2_word_pos) - 1):
                word_pos = list2_word_pos[i]
                input_seq = encoded_sentence[:, :word_pos].transpose(0, 1)  # (word_pos, batch_size)
                cache = model.init_cache(input_seq, 1)
                output, _ = model(input_seq, cache, 1, temperature, True)
                log_probs = torch.nn.functional.log_softmax(output, dim=-1)  # (word_pos, batch_size, vocab_size)
                lp = log_probs[-1, :, :]  # last token: (batch_size, vocab_size)
                
                correct_tokens = list1_encoded[:, i+1].unsqueeze(1)  # (batch_size, 1)
                log_prob_correct = lp.gather(1, correct_tokens)      # (batch_size, 1)
                print('correct',log_prob_correct[0])
                # For each other token in the list (except position i)
                # we want to check if the correct token's log prob is > all wrong tokens' log probs
                for j in range(list1_encoded.shape[1]):
                    
                    if j == i+1 or j==0:
                        continue
                    wrong_tokens = list1_encoded[:, j].unsqueeze(1)  # (batch_size, 1)
                    log_prob_wrong = lp.gather(1, wrong_tokens)      # (batch_size, 1)
                    print('wrong', log_prob_wrong[0])
                    # Update correctness per sequence: 
                    # correct_sequences stays True only if correct token prob > wrong token prob for all tokens and positions
                    correct_sequences &= (log_prob_correct >= log_prob_wrong).squeeze(1)

            # Convert boolean mask to float and average over batch for this list_len
            accuracy = correct_sequences.float().mean().item()
            results_by_length[list_len].append(accuracy)

    return results_by_length


In [224]:


def eval_simplified(model, dataloader, temperature):
    model.eval()
    
    results_by_length = {}
    
    with torch.no_grad():
        for batch in dataloader:
            sentence = batch['sentence']
            encoded_sentence = batch["encoded_sentence"]  
            list1_encoded = batch['list1_encoded'][:,::2]   
            list2_encoded = batch['list2_encoded'][:,::2]
            list2 = batch['list2'][0][::2]
            list1=batch['list1'][0][::2]
            list_len = list1_encoded.shape[-1] 
            results_by_length.setdefault(list_len, [])          
            marker = batch["marker"]

            list2_word_pos = [i for i,m in enumerate(marker[0]) if m==3][::2]

            if list_len != len(list2_word_pos):
                print(f"Warning: Mismatch between list1_encoded length ({list_len}) and list2_word_pos length ({len(list2_word_pos)}) for a batch. Skipping.")
                continue 
            
            batch_size, seq_len = encoded_sentence.shape
            
            correct_sequences = torch.ones(batch_size, dtype=torch.bool)  # Start with all True
            
   
            for k in range(1, list_len): 
                word_pos = list2_word_pos[k] 
                input_seq = encoded_sentence[:, :word_pos].transpose(0, 1)  
                #print(sentence[0][:word_pos])
                cache = model.init_cache(input_seq, 1)
                output, _ = model(input_seq, cache, 1, temperature, True)               
                log_probs = torch.nn.functional.log_softmax(output, dim=-1)  
                lp = log_probs[-1, :, :]  
                
                correct_tokens = list1_encoded[:, k].unsqueeze(1)  
                log_prob_correct = lp.gather(1, correct_tokens)      
                for j in range(list_len): 
                    if j == k: 
                        continue
                    if j == 0: 
                        continue
                    print(list2[j])
                    print(list2)
                    wrong_tokens = list2_encoded[:, j].unsqueeze(1)  
                    log_prob_wrong = lp.gather(1, wrong_tokens)      
                    correct_sequences &= (log_prob_correct >= log_prob_wrong).squeeze(1)

            accuracy = correct_sequences.float().mean().item()
            results_by_length[list_len].append(accuracy)

    return results_by_length


In [225]:
res=eval_simplified(model, dataloader, temperature)

roof
['window', 'door', 'roof']
door
['window', 'door', 'roof']
roof
['window', 'door', 'roof', 'wall', 'floor']
wall
['window', 'door', 'roof', 'wall', 'floor']
floor
['window', 'door', 'roof', 'wall', 'floor']
door
['window', 'door', 'roof', 'wall', 'floor']
wall
['window', 'door', 'roof', 'wall', 'floor']
floor
['window', 'door', 'roof', 'wall', 'floor']
door
['window', 'door', 'roof', 'wall', 'floor']
roof
['window', 'door', 'roof', 'wall', 'floor']
floor
['window', 'door', 'roof', 'wall', 'floor']
door
['window', 'door', 'roof', 'wall', 'floor']
roof
['window', 'door', 'roof', 'wall', 'floor']
wall
['window', 'door', 'roof', 'wall', 'floor']
roof
['window', 'door', 'roof', 'wall', 'floor', 'ceiling', 'room']
wall
['window', 'door', 'roof', 'wall', 'floor', 'ceiling', 'room']
floor
['window', 'door', 'roof', 'wall', 'floor', 'ceiling', 'room']
ceiling
['window', 'door', 'roof', 'wall', 'floor', 'ceiling', 'room']
room
['window', 'door', 'roof', 'wall', 'floor', 'ceiling', 'room']
d

In [226]:
res

{3: [0.017391303554177284], 5: [0.0], 7: [0.0], 10: [0.0]}

In [11]:
import torch
import math # For log2 conversion

def eval_surprisal(model, dataloader, temperature):
    model.eval()
    
    # Store results for each list length.
    # Each entry in the list will be a dictionary of metrics for one batch.
    results_by_length = {} 
    
    with torch.no_grad():
        for batch in dataloader:
            encoded_sentence = batch["encoded_sentence"]       # shape: (batch_size, seq_len)
            list1_encoded = batch['list1_encoded'][:,::2]   
            list_len = list1_encoded.shape[-1] 
            results_by_length.setdefault(list_len, [])
            
            marker = batch["marker"]
            list2_word_pos = [i for i,m in enumerate(marker[0]) if m==3][::2]
            
            if list_len != len(list2_word_pos):
                print(f"Warning: Mismatch between list1_encoded length ({list_len}) and list2_word_pos length ({len(list2_word_pos)}) for a batch. Skipping.")
                continue 
            
            batch_size, seq_len = encoded_sentence.shape
            
            # Initialize accumulators for surprisal metrics for the current batch
            # surprisal_ranking_correct_for_batch: True if correct word has lowest surprisal among candidates for that seq
            # total_correct_surprisal_nats_for_batch: sum of surprisals for correct words (in nats)
            # num_predictions_per_sequence: count of predictions made for each sequence in the batch
            
            surprisal_ranking_correct_for_batch = torch.ones(batch_size, dtype=torch.bool) 
            total_correct_surprisal_nats_for_batch = torch.zeros(batch_size, dtype=torch.float)
            num_predictions_per_sequence = torch.zeros(batch_size, dtype=torch.long)

            # Loop through the words to predict, starting from the second word (index 1)
            for k in range(1, list_len): 
                word_pos = list2_word_pos[k] 
                input_seq = encoded_sentence[:, :word_pos].transpose(0, 1)  # (word_pos, batch_size)
                
                cache = model.init_cache(input_seq, 1)
                output, _ = model(input_seq, cache, 1, temperature, True)
                
                # Get log_probabilities for the predicted token
                log_probs = torch.nn.functional.log_softmax(output, dim=-1) # (word_pos, batch_size, vocab_size)
                lp = log_probs[-1, :, :]  # log_probs for the last token in input_seq (batch_size, vocab_size)
                
                correct_tokens = list1_encoded[:, k].unsqueeze(1) # (batch_size, 1)
                log_prob_correct = lp.gather(1, correct_tokens)  # (batch_size, 1)

                # Calculate surprisal for the correct token
                # Surprisal (nats) = -log_e(P(token))
                surprisal_correct_nats = -log_prob_correct.squeeze(1) # (batch_size,)
                
                # Accumulate surprisal for average surprisal calculation
                total_correct_surprisal_nats_for_batch += surprisal_correct_nats
                num_predictions_per_sequence += 1 # Increment for each prediction point

                # Check if correct token's surprisal is lower than (or equal to) all wrong tokens' surprisals
                for j in range(list_len): 
                    if j == k: # Skip the correct token itself
                        continue
                    # You can also choose to exclude the first element (index 0) from being a "wrong" choice if it's never a target.
                    # if j == 0:
                    #     continue

                    wrong_tokens = list1_encoded[:, j].unsqueeze(1) # (batch_size, 1)
                    log_prob_wrong = lp.gather(1, wrong_tokens)      # (batch_size, 1)
                    
                    # (log_prob_correct >= log_prob_wrong) is equivalent to (-surprisal_correct <= -surprisal_wrong)
                    # which simplifies to (surprisal_correct <= surprisal_wrong)
                    surprisal_ranking_correct_for_batch &= (log_prob_correct >= log_prob_wrong).squeeze(1)

            # --- Calculate batch-level metrics ---
            
            # 1. Surprisal Ranking Accuracy (similar to your original correctness)
            # What percentage of sequences had the correct word less surprising than all others for ALL prediction points?
            batch_surprisal_ranking_accuracy = surprisal_ranking_correct_for_batch.float().mean().item()

            # 2. Average Surprisal of Correct Words (in nats and bits)
            # Handle cases where num_predictions_per_sequence might be 0 (e.g., list_len < 2)
            valid_sequences = num_predictions_per_sequence > 0
            
            if valid_sequences.any():
                avg_correct_surprisal_nats_per_sequence = torch.where(
                    valid_sequences,
                    total_correct_surprisal_nats_for_batch / num_predictions_per_sequence.float(),
                    torch.tensor(float('nan'), device=total_correct_surprisal_nats_for_batch.device)
                )
                batch_avg_correct_surprisal_nats = avg_correct_surprisal_nats_per_sequence[valid_sequences].mean().item()
                batch_avg_correct_surprisal_bits = batch_avg_correct_surprisal_nats / math.log(2)
            else:
                batch_avg_correct_surprisal_nats = float('nan')
                batch_avg_correct_surprisal_bits = float('nan')

            results_by_length[list_len].append({
                'surprisal_ranking_accuracy': batch_surprisal_ranking_accuracy,
                'avg_correct_surprisal_nats': batch_avg_correct_surprisal_nats,
                'avg_correct_surprisal_bits': batch_avg_correct_surprisal_bits
            })

    # --- Aggregate results across all batches for each list length ---
    final_aggregated_results = {}
    for list_len, list_of_batch_metrics in results_by_length.items():
        if not list_of_batch_metrics:
            continue
        
        # Filter out NaN values before averaging if any batch had issues (e.g., no predictions)
        filtered_metrics = [m for m in list_of_batch_metrics if not math.isnan(m['avg_correct_surprisal_nats'])]

        if not filtered_metrics: # All batches for this list_len were invalid
            final_aggregated_results[list_len] = {
                'surprisal_ranking_accuracy': float('nan'),
                'avg_correct_surprisal_nats': float('nan'),
                'avg_correct_surprisal_bits': float('nan')
            }
            continue

        total_ranking_accuracy = sum(m['surprisal_ranking_accuracy'] for m in filtered_metrics)
        total_avg_correct_surprisal_nats = sum(m['avg_correct_surprisal_nats'] for m in filtered_metrics)
        total_avg_correct_surprisal_bits = sum(m['avg_correct_surprisal_bits'] for m in filtered_metrics)
        
        count = len(filtered_metrics)
        final_aggregated_results[list_len] = {
            'surprisal_ranking_accuracy': total_ranking_accuracy / count,
            'avg_correct_surprisal_nats': total_avg_correct_surprisal_nats / count,
            'avg_correct_surprisal_bits': total_avg_correct_surprisal_bits / count
        }
    
    return final_aggregated_results

In [12]:
res = eval_surprisal(model, dataloader, temperature)

In [13]:
res

{3: {'surprisal_ranking_accuracy': 0.008695651777088642,
  'avg_correct_surprisal_nats': 5.6813225746154785,
  'avg_correct_surprisal_bits': 8.19641590408827},
 5: {'surprisal_ranking_accuracy': 0.0,
  'avg_correct_surprisal_nats': 5.374322414398193,
  'avg_correct_surprisal_bits': 7.753508295390675},
 7: {'surprisal_ranking_accuracy': 0.0,
  'avg_correct_surprisal_nats': 5.2306013107299805,
  'avg_correct_surprisal_bits': 7.546162571857455},
 10: {'surprisal_ranking_accuracy': 0.0,
  'avg_correct_surprisal_nats': 5.118319988250732,
  'avg_correct_surprisal_bits': 7.38417486473219}}

In [21]:
from language_models.model import RNNModel as lstm
from language_models.utils import repackage_hidden
checkpoint = '/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam_full_check_shuffled/epoch_40.pt'
data_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data"
dictionary = Dictionary(data_path)
model_lstm = lstm("LSTM", len(dictionary), 650, 650, 2, 0.2, False).to(device)

In [18]:


def eval_simplified(model, dataloader, temperature):
    model.eval()
    
    results_by_length = {}
    batch_size=230
    hidden = model.init_hidden(batch_size)
    with torch.no_grad():
        for batch in dataloader:
            sentence = batch['sentence']
            encoded_sentence = batch["encoded_sentence"]  
            list1_encoded = batch['list1_encoded'][:,::2]   
            list2_encoded = batch['list2_encoded'][:,::2]
            list2 = batch['list2'][0][::2]
            list1=batch['list1'][0][::2]
            list_len = list1_encoded.shape[-1] 
            results_by_length.setdefault(list_len, [])          
            marker = batch["marker"]

            list2_word_pos = [i for i,m in enumerate(marker[0]) if m==3][::2]

            if list_len != len(list2_word_pos):
                print(f"Warning: Mismatch between list1_encoded length ({list_len}) and list2_word_pos length ({len(list2_word_pos)}) for a batch. Skipping.")
                continue 
            
            batch_size, seq_len = encoded_sentence.shape
            
            correct_sequences = torch.ones(batch_size, dtype=torch.bool)  # Start with all True
            
   
            for k in range(1, list_len): 
                word_pos = list2_word_pos[k] 
                input_seq = encoded_sentence[:, :word_pos].transpose(0, 1)  
                #print(sentence[0][:word_pos])
                # cache = model.init_cache(input_seq, 1)
                # output, _ = model(input_seq, cache, 1, temperature, True)   
                output, hidden = model(input_seq,hidden)
                hidden = repackage_hidden(hidden)
                log_probs = torch.nn.functional.log_softmax(output, dim=-1)  
                lp = log_probs[-1, :, :]  
                
                correct_tokens = list1_encoded[:, k].unsqueeze(1)  
                log_prob_correct = lp.gather(1, correct_tokens)      
                for j in range(list_len): 
                    if j == k: 
                        continue
                    if j == 0: 
                        continue
                    print(list2[j])
                    print(list2)
                    wrong_tokens = list2_encoded[:, j].unsqueeze(1)  
                    log_prob_wrong = lp.gather(1, wrong_tokens)      
                    correct_sequences &= (log_prob_correct >= log_prob_wrong).squeeze(1)

            accuracy = correct_sequences.float().mean().item()
            results_by_length[list_len].append(accuracy)

    return results_by_length


In [22]:
res=eval_simplified(model_lstm, dataloader, temperature)

roof
['window', 'door', 'roof']
door
['window', 'door', 'roof']
roof
['window', 'door', 'roof', 'wall', 'floor']
wall
['window', 'door', 'roof', 'wall', 'floor']
floor
['window', 'door', 'roof', 'wall', 'floor']
door
['window', 'door', 'roof', 'wall', 'floor']
wall
['window', 'door', 'roof', 'wall', 'floor']
floor
['window', 'door', 'roof', 'wall', 'floor']
door
['window', 'door', 'roof', 'wall', 'floor']
roof
['window', 'door', 'roof', 'wall', 'floor']
floor
['window', 'door', 'roof', 'wall', 'floor']
door
['window', 'door', 'roof', 'wall', 'floor']
roof
['window', 'door', 'roof', 'wall', 'floor']
wall
['window', 'door', 'roof', 'wall', 'floor']
roof
['window', 'door', 'roof', 'wall', 'floor', 'ceiling', 'room']
wall
['window', 'door', 'roof', 'wall', 'floor', 'ceiling', 'room']
floor
['window', 'door', 'roof', 'wall', 'floor', 'ceiling', 'room']
ceiling
['window', 'door', 'roof', 'wall', 'floor', 'ceiling', 'room']
room
['window', 'door', 'roof', 'wall', 'floor', 'ceiling', 'room']
d

In [23]:
res

{3: [0.008695651777088642], 5: [0.0], 7: [0.0], 10: [0.0]}